In [1]:
from pathlib import Path
import duckdb

DAY = Path("../data/sample/2026-04-01.csv").resolve()
print(DAY, DAY.exists())

con = duckdb.connect()

/Users/pranavaditya/projects/backblaze-drive-stats-platform/data/sample/2026-04-01.csv True


In [2]:
con.sql(f"DESCRIBE SELECT * FROM read_csv('{DAY}')").df().head(15)

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,serial_number,VARCHAR,YES,None,None,None
2,model,VARCHAR,YES,None,None,None
3,capacity_bytes,BIGINT,YES,None,None,None
4,failure,BIGINT,YES,None,None,None
5,datacenter,VARCHAR,YES,None,None,None
6,cluster_id,VARCHAR,YES,None,None,None
7,vault_id,BIGINT,YES,None,None,None
8,pod_id,VARCHAR,YES,None,None,None
9,pod_slot_num,VARCHAR,YES,None,None,None


In [3]:
#Uniqueness check
con.sql(f"""
    SELECT
        count(*)                      AS rows,
        count(DISTINCT serial_number) AS distinct_drives
    FROM read_csv('{DAY}')
""").df()

,rows,distinct_drives
0,345738,345738


In [4]:
con.sql(f"""
    SELECT serial_number, model, capacity_bytes
    FROM read_csv('{DAY}')
    WHERE failure = 1
""").df()

,serial_number,model,capacity_bytes
0,7060A0NKFV8G,TOSHIBA MG08ACA16TEY,16000900661248
1,2AGMM03Y,HGST HUH721212ALN604,12000138625024
2,2CKPN55J,WDC WUH721816ALE6L4,16000900661248
3,1QHP24LX,WDC WUH722222ALE6L4,22000969973760
4,88R0A0CJF97G,TOSHIBA MG07ACA14TA,14000519643136
5,2BJB9JYD,WDC WUH721816ALE6L0,16000900661248
6,88Q0A08HF97G,TOSHIBA MG07ACA14TA,14000519643136
7,60P0A03LFV8G,TOSHIBA MG08ACA16TEY,16000900661248
8,3520A0DWF4MJ,TOSHIBA MG10ACA20TE,20000588955648
9,12B0A06FFVKG,TOSHIBA MG08ACA16TA,16000900661248


In [5]:
con.sql(f"""
    SELECT model, count(*) AS drives
    FROM read_csv('{DAY}')
    GROUP BY model
    ORDER BY drives DESC
    LIMIT 10
""").df()

,model,drives
0,WDC WUH722222ALE6L4,45570
1,TOSHIBA MG08ACA16TA,39747
2,TOSHIBA MG07ACA14TA,37152
3,ST16000NM001G,34588
4,WDC WUH721816ALE6L4,26261
5,TOSHIBA MG10ACA20TE,20406
6,ST12000NM0008,18461
7,ST12000NM001G,13163
8,HGST HUH721212ALE604,13109
9,ST8000NM0055,13045


In [6]:
#Completeness Check
con.sql(f"""
    SELECT
        count(*) - count(datacenter)   AS datacenter_empty,
        count(*) - count(cluster_id)   AS cluster_id_empty,
        count(*) - count(vault_id)     AS vault_id_empty,
        count(*) - count(pod_id)       AS pod_id_empty,
        count(*) - count(model)        AS model_empty,
        sum(CASE WHEN capacity_bytes <= 0 THEN 1 ELSE 0 END) AS bad_capacity
    FROM read_csv('{DAY}')
""").df()

,datacenter_empty,cluster_id_empty,vault_id_empty,pod_id_empty,model_empty,bad_capacity
0,0,0,0,0,0,0.0


In [7]:
summary = con.sql(f"SUMMARIZE SELECT * FROM read_csv('{DAY}')").df()
summary[["column_name", "column_type", "null_percentage"]].head(20)

,column_name,column_type,null_percentage
0,date,DATE,0.00
1,serial_number,VARCHAR,0.00
2,model,VARCHAR,0.00
3,capacity_bytes,BIGINT,0.00
4,failure,BIGINT,0.00
5,datacenter,VARCHAR,0.00
6,cluster_id,VARCHAR,0.00
7,vault_id,BIGINT,0.00
8,pod_id,VARCHAR,0.00
9,pod_slot_num,VARCHAR,1.17


In [8]:
smart = summary[summary["column_name"].str.startswith("smart_")]
print("SMART columns:", len(smart))
print("More than 90% empty:", (smart["null_percentage"].astype(float) > 90).sum())
smart.sort_values("null_percentage").head(10)[["column_name", "null_percentage"]]

SMART columns: 186
More than 90% empty: 112


,column_name,null_percentage
31,smart_12_normalized,0.13
120,smart_194_raw,0.13
119,smart_194_normalized,0.13
32,smart_12_raw,0.13
26,smart_9_raw,0.13
25,smart_9_normalized,0.13
11,smart_1_normalized,0.22
12,smart_1_raw,0.22
116,smart_192_raw,0.45
115,smart_192_normalized,0.45


In [9]:
con.sql(f"""
    SELECT 'pod_id' AS col, pod_id AS value, count(*) AS n
    FROM read_csv('{DAY}')
    WHERE pod_id IS NOT NULL AND TRY_CAST(pod_id AS BIGINT) IS NULL
    GROUP BY ALL

    UNION ALL
    SELECT 'pod_slot_num', pod_slot_num, count(*)
    FROM read_csv('{DAY}')
    WHERE pod_slot_num IS NOT NULL AND TRY_CAST(pod_slot_num AS BIGINT) IS NULL
    GROUP BY ALL

    UNION ALL
    SELECT 'cluster_id', cluster_id, count(*)
    FROM read_csv('{DAY}')
    WHERE cluster_id IS NOT NULL AND TRY_CAST(cluster_id AS BIGINT) IS NULL
    GROUP BY ALL

    ORDER BY n DESC
    LIMIT 20
""").df()

,col,value,n


In [10]:
for col in ["datacenter", "failure", "is_legacy_format"]:
    print(con.sql(f"""
        SELECT {col}, count(*) AS n
        FROM read_csv('{DAY}')
        GROUP BY ALL
        ORDER BY n DESC
    """).df(), "\n")

  datacenter       n
0       phx1  109708
1       sac0   76996
2       sac2   73514
3       iad1   52701
4       ams5   29219
5       yyz1    3600 

   failure       n
0        0  345724
1        1      14 

   is_legacy_format       n
0             False  345738 



In [11]:
con.sql(f"""
    SELECT DISTINCT pod_id, pod_slot_num, cluster_id
    FROM read_csv('{DAY}')
    LIMIT 10
""").df()

,pod_id,pod_slot_num,cluster_id
0,02,0012,000
1,18,0054,000
2,16,0039,000
3,19,0059,000
4,08,0001,031
5,19,0002,000
6,04,0040,000
7,12,0051,000
8,08,0026,050
9,07,0051,000
